part 4 of project flights:

In [72]:
# import and connect data
import sqlite3
import pandas as pd

con = sqlite3.connect("flights_database.db")

In [73]:
# load the tabels 
flights = pd.read_sql("""
SELECT *
FROM flights;
""", con)

airports = pd.read_sql("SELECT * FROM airports;", con)
planes = pd.read_sql("SELECT * FROM planes;", con)
weather = pd.read_sql("SELECT * FROM weather;", con)

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2023,1,1,1.0,2038,203.0,328.0,3,205.0,UA,628,N25201,EWR,SMF,367.0,2500.0,20.0,38.0,1.672603e+09
1,2023,1,1,18.0,2300,78.0,228.0,135,53.0,DL,393,N830DN,JFK,ATL,108.0,760.0,23.0,0.0,1.672614e+09
2,2023,1,1,31.0,2344,47.0,500.0,426,34.0,B6,371,N807JB,JFK,BQN,190.0,1576.0,23.0,44.0,1.672614e+09
3,2023,1,1,33.0,2140,173.0,238.0,2352,166.0,B6,1053,N265JB,JFK,CHS,108.0,636.0,21.0,40.0,1.672607e+09
4,2023,1,1,36.0,2048,228.0,223.0,2252,211.0,UA,219,N17730,EWR,DTW,80.0,488.0,20.0,48.0,1.672603e+09


In [74]:
# check missing values 
missing = flights.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(flights) * 100).round(2)

pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})

,missing_count,missing_percent
arr_delay,12534,2.88
air_time,12534,2.88
arr_time,11453,2.63
dep_time,10738,2.47
dep_delay,10738,2.47
tailnum,1913,0.44
year,0,0.00
origin,0,0.00
minute,0,0.00
hour,0,0.00


Around 2–3% of the observations have missing values in actual departure/arrival times and delays.
These likely correspond to cancelled or diverted flights.
We therefore keep these rows but create a cancelled indicator variable.

In [75]:
# handle missing values 
# Create cancelled indicator
flights["cancelled"] = flights["dep_time"].isna()

# Example: drop rows missing critical variables
flights_clean = flights.dropna(subset=["sched_dep_time", "sched_arr_time"])

chech duplicates:

In [76]:
flight_id_cols = [
    "year","month","day",
    "carrier","flight",
    "origin","dest",
    "sched_dep_time"
]

duplicates = flights.duplicated(subset=flight_id_cols, keep=False)
duplicates.sum()

np.int64(0)

No duplicate flights were found when defining a flight uniquely by year, month, day, carrier, flight number, origin, destination, and scheduled departure time.

inspect duplicates:

In [77]:
flights[duplicates].sort_values(flight_id_cols).head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,cancelled


remove duplicates:

In [78]:
flights = flights.drop_duplicates(subset=flight_id_cols)

convert times to dataframe

In [79]:
# date column
flights["date"] = pd.to_datetime(dict(
    year=flights.year,
    month=flights.month,
    day=flights.day
))

In [80]:
#HHMM to datetime
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    hh = x // 100
    mm = x % 100
    return hh*60 + mm

def make_datetime(date, hhmm):
    mins = hhmm.map(hhmm_to_minutes)
    return date + pd.to_timedelta(mins, unit="m")

flights["sched_dep_dt"] = make_datetime(flights["date"], flights["sched_dep_time"])
flights["dep_dt"] = make_datetime(flights["date"], flights["dep_time"])
flights["sched_arr_dt"] = make_datetime(flights["date"], flights["sched_arr_time"])
flights["arr_dt"] = make_datetime(flights["date"], flights["arr_time"])

check if flights are in order:

In [81]:
# consistency 
# Compute corrected arrival time using air_time
flights["arr_dt_corrected"] = (
    flights["dep_dt"] + pd.to_timedelta(flights["air_time"], unit="m")
)

# Check difference
difference = (
    (flights["arr_dt_corrected"] - flights["dep_dt"]).dt.total_seconds() / 60
    - flights["air_time"]
).abs()

(difference > 5).sum()

np.int64(0)

We verified that arrival time equals departure time plus air_time.
After correcting for overnight arrivals, no inconsistencies were found.

Thus, the time variables are internally consistent.

resolve? 

In [82]:
flights["arr_dt_corrected"] = flights["dep_dt"] + pd.to_timedelta(flights["air_time"], unit="m")

local arrival time:

In [83]:
# timezone info
airports.columns

Index(['faa', 'name', 'lat', 'lon', 'alt', 'tz', 'dst', 'tzone'], dtype='object')

In [85]:
# 1) Localize corrected arrival time as NYC time (handles DST safely)
arr_ny = pd.to_datetime(flights["arr_dt_corrected"]).dt.tz_localize(
    "America/New_York",
    nonexistent="shift_forward",
    ambiguous="NaT"
)

# 2) Convert per-destination timezone (still needs mapping; apply just for conversion)
tz_map = airports.set_index("faa")["tzone"].to_dict()

def convert_dest(dt, dest):
    tzname = tz_map.get(dest)
    if pd.isna(dt) or tzname is None or pd.isna(tzname):
        return pd.NaT
    try:
        return dt.tz_convert(tzname)
    except Exception:
        return pd.NaT

flights["arr_local_time"] = [convert_dest(dt, d) for dt, d in zip(arr_ny, flights["dest"])]

wind7precipitation and plane type:

In [ ]:
# weather 
weather["time_hour"] = pd.to_datetime(weather["time_hour"])

# Make sure flights hour is also naive datetime (no timezone) rounded to hour
flights["hour_key"] = pd.to_datetime(flights["sched_dep_dt"]).dt.floor("h")

flights_weather = flights.merge(
    weather,
    left_on=["origin", "hour_key"],
    right_on=["origin", "time_hour"],
    how="left"
)

# quick check: what fraction matched?
flights_weather["wind_speed"].notna().mean()

In [ ]:
# analysis brief
summary = flights_weather.groupby("carrier").agg(
    avg_delay=("arr_delay","mean"),
    avg_wind=("wind_speed","mean"),
    avg_precip=("precip","mean"),
    n=("arr_delay","size")
).sort_values("avg_delay", ascending=False)

summary

dashboard:

In [ ]:
# airports
def airport_stats(df, origin=None, dest=None):
    x = df.copy()
    
    if origin:
        x = x[x["origin"] == origin]
    if dest:
        x = x[x["dest"] == dest]
        
    return {
        "n_flights": len(x),
        "avg_dep_delay": x["dep_delay"].mean(),
        "avg_arr_delay": x["arr_delay"].mean(),
        "most_common_dest": x["dest"].value_counts().idxmax()
    }

In [ ]:
# visualize 
def airport_stats(df, origin=None, dest=None):
    x = df.copy()
    
    if origin:
        x = x[x["origin"] == origin]
    if dest:
        x = x[x["dest"] == dest]
        
    return {
        "n_flights": len(x),
        "avg_dep_delay": x["dep_delay"].mean(),
        "avg_arr_delay": x["arr_delay"].mean(),
        "most_common_dest": x["dest"].value_counts().idxmax()
    }

results:

In [ ]:
# worst airports by delay
delay_stats = flights.groupby("dest").agg(
    avg_delay=("arr_delay","mean"),
    n=("arr_delay","size")
)

# Keep only airports with enough flights
delay_stats = delay_stats[delay_stats["n"] > 200]

delay_stats.sort_values("avg_delay", ascending=False).head(10)

Several destinations such as PSE, ABQ, and ONT show the highest
average arrival delays among airports with sufficient flight volume.

This suggests route-specific or regional delay patterns.

In [ ]:
# distance and delay?
filtered = flights[(flights["arr_delay"] < 300) & (flights["arr_delay"] > -50)]

plt.figure()
plt.scatter(filtered["distance"], filtered["arr_delay"], alpha=0.2)
plt.xlabel("Distance")
plt.ylabel("Arrival delay")
plt.show()

flights["distance"].corr(flights["arr_delay"])

The correlation between flight distance and arrival delay is approximately 0.015,
indicating virtually no linear relationship.

Thus, longer flights are not systematically more delayed.